In [ ]:
filename = "noisy_16.txt"

x = []
y = []

with open(filename, "r") as file:
    for line in file:
        values = line.strip().split()

        if len(values) == 2:
            x.append(float(values[0]))
            y.append(float(values[1]))

print("Number of samples:", len(x))
print("First 5 x values:", x[:5])
print("First 5 y values:", y[:5])

In [ ]:
# Here we Shuffle the data
import random

data = list(zip(x, y))

random.seed(42) # Keeps the randomness constant everytime you run it
random.shuffle(data)

x = [row[0] for row in data]
y = [row[1] for row in data]

print("First 5 shuffled samples:")

for i in range(5):
    print(x[i], y[i])

In [ ]:
# Splitting of datasets of test,train and validation 

# Test : 60
# Train : 20
# Validation : 20

n = len(x)

train_end = int(0.60 * n)
test_end = int(0.80 * n)

x_train = x[:train_end]
y_train = y[:train_end]

x_test = x[train_end:test_end]
y_test = y[train_end:test_end]

x_val = x[test_end:]
y_val = y[test_end:]

print("Training samples:", len(x_train))
print("Test samples:", len(x_test))
print("Validation samples:", len(x_val))

In [ ]:
# Scaling of dataset
x_train_scaled = [value / 100.0 for value in x_train]
x_test_scaled = [value / 100.0 for value in x_test]
x_val_scaled = [value / 100.0 for value in x_val]

print("Original x:", x_train[:5])
print("Scaled x:", x_train_scaled[:5])

In [ ]:
# Beginning of Regression

def create_polynomial_matrix(x, degree):

    X = []

    for value in x:

        row = []

        for power in range(degree + 1):
            row.append(value ** power)

        X.append(row)

    return X


# X = create_polynomial_matrix([0.5, 0.2, -0.4], 3)

# for row in X:
#     print(row)

In [ ]:
def transpose(A):

    rows = len(A)
    cols = len(A[0])

    result = []

    for j in range(cols):

        row = []

        for i in range(rows):
            row.append(A[i][j])

        result.append(row)

    return result

In [ ]:
# Matrix Multiplication

def matrix_multiply(A, B):

    rows_A = len(A)
    cols_A = len(A[0])

    rows_B = len(B)
    cols_B = len(B[0])

    if cols_A != rows_B:
        raise ValueError("Matrix dimensions do not match")

    result = []

    for i in range(rows_A):

        row = []

        for j in range(cols_B):

            total = 0.0

            for k in range(cols_A):
                total += A[i][k] * B[k][j]

            row.append(total)

        result.append(row)

    return result

In [ ]:
def matrix_inverse(A):

    n = len(A)

    # Create augmented matrix [A | I]
    augmented = []

    for i in range(n):

        row = []

        for j in range(n):
            row.append(float(A[i][j]))

        for j in range(n):
            if i == j:
                row.append(1.0)
            else:
                row.append(0.0)

        augmented.append(row)

    # Gauss-Jordan elimination
    for i in range(n):

        # Find pivot
        pivot_row = i

        for r in range(i + 1, n):
            if abs(augmented[r][i]) > abs(augmented[pivot_row][i]):
                pivot_row = r

        # Check if matrix is singular
        if abs(augmented[pivot_row][i]) < 1e-12:
            raise ValueError("Matrix is singular")

        # Swap rows
        augmented[i], augmented[pivot_row] = \
            augmented[pivot_row], augmented[i]

        # Normalize pivot row
        pivot = augmented[i][i]

        for j in range(2 * n):
            augmented[i][j] /= pivot

        # Eliminate other rows
        for r in range(n):

            if r != i:

                factor = augmented[r][i]

                for j in range(2 * n):
                    augmented[r][j] -= factor * augmented[i][j]

    # Extract inverse matrix
    inverse = []

    for i in range(n):

        row = []

        for j in range(n, 2 * n):
            row.append(augmented[i][j])

        inverse.append(row)

    return inverse

In [ ]:
# Implementation of Polynomial Regression 
# β=(XT X)−1  (XT y)

def polynomial_regression(x, y, degree):

    # Create polynomial feature matrix
    X = create_polynomial_matrix(x, degree)

    # Convert y into column matrix
    Y = []

    for value in y:
        Y.append([value])

    # X transpose
    XT = transpose(X)

    # X^T X
    XTX = matrix_multiply(XT, X)

    # (X^T X)^-1
    XTX_inverse = matrix_inverse(XTX)

    # X^T y
    XTY = matrix_multiply(XT, Y)

    # beta = (X^T X)^-1 X^T y
    beta = matrix_multiply(XTX_inverse, XTY)

    # Convert [[b0], [b1], ...] into [b0, b1, ...]
    coefficients = []

    for row in beta:
        coefficients.append(row[0])

    return coefficients

In [ ]:
# Prediction of Most optimal Coefficients
def predict(x, coefficients):

    predictions = []

    for value in x:

        prediction = 0.0

        for power in range(len(coefficients)):

            prediction += coefficients[power] * (value ** power)

        predictions.append(prediction)

    return predictions

In [ ]:
# Mean Square Error calculation and measuring how good the polynomial is.

def mean_squared_error(y_true, y_pred):

    total = 0.0

    for actual, predicted in zip(y_true, y_pred):

        error = actual - predicted
        total += error ** 2

    return total / len(y_true)

In [ ]:
#Test different polynomial degrees
degrees = range(1, 11)

results = []

for degree in degrees:

    coefficients = polynomial_regression(
        x_train_scaled,
        y_train,
        degree
    )

    train_predictions = predict(
        x_train_scaled,
        coefficients
    )

    test_predictions = predict(
        x_test_scaled,
        coefficients
    )

    train_mse = mean_squared_error(
        y_train,
        train_predictions
    )

    test_mse = mean_squared_error(
        y_test,
        test_predictions
    )

    results.append((degree, train_mse, test_mse))

    print(
        "Degree:", degree,
        "| Train MSE:", train_mse,
        "| Test MSE:", test_mse
    )

In [ ]:
# Select the best degree
best_degree = results[0][0]
best_test_mse = results[0][2]

for degree, train_mse, test_mse in results:

    if test_mse < best_test_mse:

        best_degree = degree
        best_test_mse = test_mse

print("Best polynomial degree:", best_degree)
print("Best test MSE:", best_test_mse)

In [ ]:
# Final validation evaluation
best_coefficients = polynomial_regression(
    x_train_scaled,
    y_train,
    best_degree
)

validation_predictions = predict(
    x_val_scaled,
    best_coefficients
)

validation_mse = mean_squared_error(
    y_val,
    validation_predictions
)

print("Selected degree:", best_degree)
print("Validation MSE:", validation_mse)

In [ ]:
#--------------------------ADDITIONAL EXPERIMENTS OF POLYNOMIAL REGRESSION ASSIGNMENT-----------------------------#

In [ ]:
# EXPERIMENT - 1
# Visual comparison of polynomial fits

import matplotlib.pyplot as plt

# Degrees we want to compare
plot_degrees = [1, 3, 7, 9, 10, 15]

# Sort training data for a smooth curve
sorted_data = sorted(zip(x_train_scaled, y_train))

plot_x = [item[0] for item in sorted_data]
plot_y = [item[1] for item in sorted_data]

plt.figure(figsize=(12, 6))

# Plot noisy training data
plt.scatter(
    plot_x,
    plot_y,
    s=3,
    alpha=0.25,
    label="Training data"
)

# Generate smooth x values
curve_x = []

for i in range(-200, 201):
    curve_x.append(i / 200.0)

# Plot each polynomial
for degree in plot_degrees:

    coefficients = polynomial_regression(
        x_train_scaled,
        y_train,
        degree
    )

    curve_y = predict(
        curve_x,
        coefficients
    )

    plt.plot(
        curve_x,
        curve_y,
        label=f"Degree {degree}"
    )

plt.xlabel("Scaled x")
plt.ylabel("y")
plt.title("Comparison of Polynomial Fits")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# EXPERIMENT 2
# Training MSE vs Test MSE

degree_values = []
train_errors = []
test_errors = []

for result in results:

    degree_values.append(result[0])
    train_errors.append(result[1])
    test_errors.append(result[2])

plt.figure(figsize=(10, 6))

plt.plot(
    degree_values,
    train_errors,
    marker="o",
    label="Training MSE"
)

plt.plot(
    degree_values,
    test_errors,
    marker="o",
    label="Test MSE"
)

plt.xlabel("Polynomial Degree")
plt.ylabel("Mean Squared Error")
plt.title("Training and Test MSE vs Polynomial Degree")

plt.xticks(degree_values)
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
#EXPERIMENT 3
# Residual analysis on validation data

residuals = []

for actual, predicted in zip(
    y_val,
    validation_predictions
):

    residual = actual - predicted
    residuals.append(residual)


# Calculate residual statistics
residual_mean = sum(residuals) / len(residuals)

residual_min = min(residuals)
residual_max = max(residuals)


print("Residual Analysis")
print("-----------------")
print("Mean residual :", residual_mean)
print("Minimum       :", residual_min)
print("Maximum       :", residual_max)


In [ ]:
# Plot residuals

plt.figure(figsize=(10, 6))

plt.scatter(
    x_val,
    residuals,
    s=4
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Original x")
plt.ylabel("Residual = Actual - Predicted")
plt.title(
    f"Residual Plot for Degree {best_degree}"
)

plt.grid(True)

plt.show()

In [ ]:
# EXPERIMENT 4
# Learning curve

training_sizes = [500,
    1000,
    2000,
    3000,
    4000,
    5000,
    6000
]

learning_train_mse = []
learning_test_mse = []


for size in training_sizes:

    # Train using only the first 'size' samples
    coefficients = polynomial_regression(
        x_train_scaled[:size],
        y_train[:size],
        best_degree
    )

    # Training predictions
    train_predictions = predict(
        x_train_scaled[:size],
        coefficients
    )

    # Test predictions
    test_predictions = predict(
        x_test_scaled,
        coefficients
    )

    # Calculate errors
    train_error = mean_squared_error(
        y_train[:size],
        train_predictions
    )

    test_error = mean_squared_error(
        y_test,
        test_predictions
    )

    learning_train_mse.append(train_error)
    learning_test_mse.append(test_error)

    print(
        "Training samples:", size,
        "| Train MSE:", train_error,
        "| Test MSE:", test_error
    )

In [ ]:
# Plot learning curve

plt.figure(figsize=(10, 6))

plt.plot(
    training_sizes,
    learning_train_mse,
    marker="o",
    label="Training MSE"
)

plt.plot(
    training_sizes,
    learning_test_mse,
    marker="o",
    label="Test MSE"
)

plt.xlabel("Number of Training Samples")
plt.ylabel("Mean Squared Error")
plt.title(
    f"Learning Curve - Polynomial Degree {best_degree}"
)

plt.legend()
plt.grid(True)

plt.show()